# Phase 7: Build ETKDG 3D Graphs for 300k Dataset (Kaggle CPU)

## Setup
1. Upload `phase7_chonsfcl_mw200_1000_300k.csv` as Kaggle dataset (e.g. `molgap-300k-csv`)
2. Accelerator: **None** (CPU only, no GPU needed)
3. Run all cells
4. Download output `pyg_3d_graphs_etkdg_300k.pt` → use as input for Optuna notebook

In [ ]:
!pip install -q torch-geometric rdkit-pypi
!pip install -q pyg-lib -f https://data.pyg.org/whl/torch-$(python -c "import torch; print(torch.__version__.split('+')[0])")+cpu.html

In [ ]:
import os, time, json
import numpy as np
import pandas as pd
import torch
from multiprocessing import Pool, cpu_count

print(f"CPU cores: {cpu_count()}")
print(f"torch {torch.__version__}")

In [ ]:
# ══════════════════════════════════════════════════════════
# CONFIG
# ══════════════════════════════════════════════════════════
CSV_PATH = "/kaggle/input/molgap-300k-csv/phase7_chonsfcl_mw200_1000_300k.csv"
OUTPUT_DIR = "/kaggle/working"
OUT_PT = os.path.join(OUTPUT_DIR, "pyg_3d_graphs_etkdg_300k.pt")

TARGET_COLS = ["homo", "lumo", "gap"]
N_WORKERS = max(1, cpu_count() - 1)  # leave 1 core for OS
CHUNK_SIZE = 500  # molecules per worker batch

print(f"Workers: {N_WORKERS}")

In [ ]:
# ── Load CSV ──
df = pd.read_csv(CSV_PATH)
for col in TARGET_COLS + ["mw"]:
    df[col] = pd.to_numeric(df[col], errors="coerce")
df = df.dropna(subset=TARGET_COLS + ["smiles"])
df = df[df["gap"] > 0].reset_index(drop=True)
print(f"Loaded {len(df)} molecules")
print(f"MW range: {df['mw'].min():.0f} - {df['mw'].max():.0f}")
print(f"Columns: {list(df.columns)}")

In [ ]:
# ── Graph building function (runs in worker process) ──
def build_one(args):
    """Build a single PyG Data from (smiles, targets). Returns Data or None."""
    smi, targets = args
    try:
        from rdkit import Chem
        from rdkit.Chem import AllChem
        from torch_geometric.data import Data

        mol = Chem.MolFromSmiles(smi)
        if mol is None:
            return None
        mol_h = AllChem.AddHs(mol)

        # ETKDG embedding with retry
        for _ in range(2):
            if AllChem.EmbedMolecule(mol_h, AllChem.ETKDGv3()) == 0:
                break
        else:
            return None

        # MMFF force field optimization
        try:
            AllChem.MMFFOptimizeMolecule(mol_h, maxIters=200)
        except Exception:
            pass

        n = mol_h.GetNumAtoms()
        if n == 0:
            return None

        conf = mol_h.GetConformer()
        z = torch.tensor([mol_h.GetAtomWithIdx(i).GetAtomicNum() for i in range(n)], dtype=torch.long)
        pos = torch.tensor(conf.GetPositions(), dtype=torch.float32)

        # Gasteiger charges
        AllChem.ComputeGasteigerCharges(mol_h)
        charges = []
        for atom in mol_h.GetAtoms():
            c = atom.GetDoubleProp('_GasteigerCharge')
            charges.append(0.0 if (c != c) or abs(c) > 1e6 else c)

        data = Data(
            z=z,
            pos=pos,
            charges=torch.tensor(charges, dtype=torch.float32),
            y=torch.tensor(targets, dtype=torch.float32).unsqueeze(0),
        )
        return data
    except Exception:
        return None

In [ ]:
# ── Build graphs with multiprocessing ──
from tqdm.auto import tqdm

smiles_list = df["smiles"].tolist()
targets_list = df[TARGET_COLS].values.astype(np.float32).tolist()
work_items = list(zip(smiles_list, targets_list))

print(f"Building ETKDG 3D graphs for {len(work_items)} molecules...")
print(f"Using {N_WORKERS} workers, chunk_size={CHUNK_SIZE}")

t0 = time.time()
data_list = []
failed = 0

with Pool(N_WORKERS) as pool:
    for result in tqdm(pool.imap(build_one, work_items, chunksize=CHUNK_SIZE),
                       total=len(work_items), desc="Graphs", unit="mol"):
        if result is not None:
            data_list.append(result)
        else:
            failed += 1

elapsed = time.time() - t0
print(f"\nDone in {elapsed/60:.1f} min ({elapsed/len(work_items)*1000:.1f} ms/mol)")
print(f"Success: {len(data_list)}/{len(work_items)} ({failed} failed, {failed/len(work_items)*100:.1f}%)")

In [ ]:
# ── Save ──
print(f"Saving {len(data_list)} graphs to {OUT_PT}...")
torch.save(data_list, OUT_PT)
size_gb = os.path.getsize(OUT_PT) / 1e9
print(f"Saved: {size_gb:.2f} GB")

# Stats
n_atoms = [d.z.shape[0] for d in data_list]
print(f"\nAtom count stats:")
print(f"  mean: {np.mean(n_atoms):.0f}")
print(f"  median: {np.median(n_atoms):.0f}")
print(f"  max: {np.max(n_atoms)}")
print(f"  min: {np.min(n_atoms)}")

# Save metadata
meta = {
    "n_molecules": len(data_list),
    "n_failed": failed,
    "n_input": len(work_items),
    "build_time_min": round(elapsed / 60, 1),
    "ms_per_mol": round(elapsed / len(work_items) * 1000, 1),
    "file_size_gb": round(size_gb, 2),
    "atom_count_mean": round(float(np.mean(n_atoms)), 1),
    "atom_count_max": int(np.max(n_atoms)),
    "has_charges": True,
}
with open(os.path.join(OUTPUT_DIR, "graph_build_meta.json"), "w") as f:
    json.dump(meta, f, indent=2)
print(f"\nMetadata saved. Download pyg_3d_graphs_etkdg_300k.pt and upload as Kaggle dataset for Optuna notebook.")